In [ ]:
import json
import numpy as np
from pathlib import Path
from typing import Dict, List

from model_ranking import (
    load_h5,
    get_ckpt_eval_scores,
    find_selftraining_pred_paths,
    get_NA_prediction_path,
    MODEL_ABBREVIATIONS_TO_DATASET

)

from pytorch3dunet.unet3d.config import load_config_direct #pyright: ignore[reportUnknownVariableType]

In [22]:
approach_mapping = {
    "DO": "feature_perturbation",
    "def": "default_selftraining",
    "cfd": "confidence_threshold",
    "F1": "direct_eval",
    "torchem": "supervised_torchem",
    "_SF_": "supervised",
    "ST": "supervised_training",
}

In [23]:
models: Dict[str, List[str]] = {
    "HtoE": [
        "HtoE_def_Hm_model4_2",
        "HtoE_def_Hm_model_NA2",
        "HtoE_def_Hm_model_Res1",
        "HtoE_def_Hm_model_Unetr2",
    ],
    "RtoE": [
        "RtoE_def_Rm_model4",
        "RtoE_def_Rm_model_NA2",
        "RtoE_def_Rm_model_Res1",
        "RtoE_def_Rm_model_Unetr2",
    ],
    "VtoE": [
        "VtoE_def_V_model2",
        "VtoE_def_V_model_NA2",
        #"VtoE_def_V_model_NA2_2",
        "VtoE_def_V_model_Res1",
    ],
    "EtoH": [
        "EtoH_def_E_model5",
        "EtoH_def_E_model_NA2",
        "EtoH_def_E_model_Res1",
        "EtoH_def_E_model_Unetr2",
    ],
    "RtoH": [
        "RtoH_def_Rm_model4",
        "RtoH_def_Rm_model_NA2",
        "RtoH_def_Rm_model_Res1",
        "RtoH_def_Rm_model_Unetr2",
    ],
    "VtoH": [
        "VtoH_def_V_model2",
        "VtoH_def_V_model_NA2",
        "VtoH_def_V_model_Res1",
    ],
    "EtoR": [
        "EtoR_def_E_model5",
        "EtoR_def_E_model_NA2",
        "EtoR_def_E_model_Res1",
        "EtoR_def_E_model_Unetr2",
    ],
    "HtoR": [
        "HtoR_def_Hm_model4",
        "HtoR_def_Hm_model_NA2",
        #"HtoR_def_Hm_model_NA2_2",
        "HtoR_def_Hm_model_Res1",
        "HtoR_def_Hm_model_Unetr2",
    ],
    "VtoR": [
        "VtoR_def_V_model2",
        "VtoR_def_V_model_NA2",
        "VtoR_def_V_model_Res1",
    ],
    "EtoV": [
        "EtoV_def_E_model5",
        "EtoV_def_E_model_NA2",
        "EtoV_def_E_model_Res1",
        "EtoV_def_E_model_Unetr2",
    ],
    "HtoV": [
        "HtoV_def_Hm_model4",
        "HtoV_def_Hm_model_NA2",
        "HtoV_def_Hm_model_Res1",
        "HtoV_def_Hm_model_Unetr2",
    ],
    "RtoV": [
        "RtoV_def_Rm_model4",
        "RtoV_def_Rm_model_NA2",
        "RtoV_def_Rm_model_Res1",
        "RtoV_def_Rm_model_Unetr2",
    ],
}

In [24]:
epoch_ids = list(range(1, 6, 1))
base_transfer_path = "/scratch/talks/consistency_results/patch_segmentation/mitochondria"
run_approach= "consistency"
run_id = "P_full"

In [38]:
#per_target_perf_results: Dict[str, Dict[str, Dict[str,float]]] = {}
per_target_perf_results: Dict[str, Dict[str, float]] = {}
for transfer, model_list in models.items():
    print(f"Transfer: {transfer}")
    target = MODEL_ABBREVIATIONS_TO_DATASET[transfer[-1]]
    if target not in per_target_perf_results:
        per_target_perf_results[target] = {}
    transfer_gap = f"{MODEL_ABBREVIATIONS_TO_DATASET[transfer[0]]}_to_{target}_gap"
    for i, model_name in enumerate(model_list):
        if "SS" in model_name:
            base_path = Path("/g/kreshuk/talks/model_ranking_results/SemiSupervised-Finetuning/Mitochondria")
        elif "SF" in model_name:
            base_path = Path("/g/kreshuk/talks/model_ranking_results/Supervised-Finetuning/Mitochondria")
        elif "ST" in model_name:
            base_path = Path("/g/kreshuk/talks/model_ranking_results/Supervised-training/Mitochondria")
        else:
            base_path = Path("/g/kreshuk/talks/model_ranking_results/Self-Finetuning/Mitochondria")
        approach = None
        best_epochs_path = None

        for key in approach_mapping:
            if key in model_name:
                approach = approach_mapping[key]
                best_epochs_path = base_path / f"{approach}_best_epochs/best_epochs_per_transfer.json"

        assert approach is not None, f"Approach not found for model: {model_name}"
        assert best_epochs_path is not None, f"Best epochs path not found for model: {model_name}"
        assert best_epochs_path.exists(), f"Best epochs file not found: {best_epochs_path}"

        finetuned_pred_path = find_selftraining_pred_paths([model_name], approach=approach, base_path=base_path)[0]
        
        with open(best_epochs_path, "r") as f:
            best_epoch_id = json.load(f)[transfer_gap][model_name]
        
        if best_epoch_id > 0:
            _, median_eval_scores = get_ckpt_eval_scores(finetuned_pred_path, [best_epoch_id])
            median_eval_score = median_eval_scores[0]
        
        else:
            yaml_paths = list((finetuned_pred_path.parent.parent / "checkpoints" / model_name).glob("*.yaml"))
            assert len(yaml_paths) == 1, f"Expected one YAML file for {model_name}, found {len(yaml_paths)}"
            yaml_path = yaml_paths[0]
            initial_model_name = Path(load_config_direct(yaml_path)[0]["training"]["model_cfg"]["source_checkpoint"]).parent.stem
            initial_pred_path = get_NA_prediction_path(
                model_name=initial_model_name,
                target=target,
                base_path=base_transfer_path,
                approach=run_approach,
                run_id=run_id,
            )
            eval_score_direct = load_h5(initial_pred_path, "hard_f1")
            median_eval_score = np.median(eval_score_direct, axis=0)[1]
        # per_target_perf_results[target].update({
        #     model_name : {"norm_Normalize": median_eval_score}
        # })
        per_target_perf_results[target].update({
            model_name : median_eval_score
        })


Transfer: HtoE
Transfer: RtoE
Transfer: VtoE
Transfer: EtoH
Transfer: RtoH
Transfer: VtoH
Transfer: EtoR
Transfer: HtoR
Transfer: VtoR
Transfer: EtoH
Transfer: RtoH
Transfer: VtoH
Transfer: EtoR
Transfer: HtoR
Transfer: VtoR
Transfer: EtoV
Transfer: HtoV
Transfer: RtoV
Transfer: EtoV
Transfer: HtoV
Transfer: RtoV


In [40]:
per_target_perf_results

{'EPFL': {'HtoE_def_Hm_model4_2': 0.92275435,
  'HtoE_def_Hm_model_NA2': 0.8762152,
  'HtoE_def_Hm_model_Res1': 0.8993416,
  'HtoE_def_Hm_model_Unetr2': 0.8936962,
  'RtoE_def_Rm_model4': 0.8757689,
  'RtoE_def_Rm_model_NA2': 0.8314564,
  'RtoE_def_Rm_model_Res1': 0.8510642,
  'RtoE_def_Rm_model_Unetr2': 0.85552967,
  'VtoE_def_V_model2': 0.45598,
  'VtoE_def_V_model_NA2': 0.60543156,
  'VtoE_def_V_model_Res1': 0.6241139},
 'Hmito': {'EtoH_def_E_model5': 0.81393844,
  'EtoH_def_E_model_NA2': 0.80165744,
  'EtoH_def_E_model_Res1': 0.8298905,
  'EtoH_def_E_model_Unetr2': 0.7272179,
  'RtoH_def_Rm_model4': 0.8111774,
  'RtoH_def_Rm_model_NA2': 0.7203462,
  'RtoH_def_Rm_model_Res1': 0.8738154,
  'RtoH_def_Rm_model_Unetr2': 0.733794,
  'VtoH_def_V_model2': 0.64884204,
  'VtoH_def_V_model_NA2': 0.652706,
  'VtoH_def_V_model_Res1': 0.67161745},
 'Rmito': {'EtoR_def_E_model5': 0.8062041,
  'EtoR_def_E_model_NA2': 0.79357094,
  'EtoR_def_E_model_Res1': 0.8433246,
  'EtoR_def_E_model_Unetr2': 0.

In [ ]:
# import json
# import os
# from model_ranking import convert_numpy_types
# from typing import Any
# save_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results"
# if not os.path.exists(save_path):
#     os.makedirs(save_path)
# with open(os.path.join(save_path, "transfer_Finetuned_Def_performance_scores_with_NORM.json"), "w") as f:
#     results: Dict[str, Any] = {
#         "performance_scores": convert_numpy_types(per_target_perf_results),
#     }
#     json.dump(results, f, indent=4)